# Phát hiện gian lận THỜI GIAN THỰC — Kafka + Spark Structured Streaming

**Phần mở rộng (bonus)** của đồ án E-Commerce Fraud Detection.

Notebook này đóng vai trò **consumer**: đọc luồng giao dịch từ Kafka topic `transactions`, áp mô hình Random Forest đã train (batch) lên từng giao dịch *ngay khi nó đến*, và phát cảnh báo khi phát hiện gian lận.

**Thứ tự chạy demo:**
1. `docker compose up -d` — khởi động Kafka
2. Chạy notebook batch [fraud_detection_sparkml.ipynb](../fraud_detection_sparkml.ipynb) tới cell cuối để lưu model lên HDFS (`hdfs://localhost:9090/models/fraud_detection_rf`)
3. `python export_data.py` — xuất dữ liệu HDFS ra `data/transactions.csv`
4. Chạy **notebook này** tới cell `start()` để bắt đầu lắng nghe
5. `python producer.py` — bơm giao dịch vào Kafka và quan sát cảnh báo

## 1. Khởi tạo Spark Session (kèm connector Kafka)

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, hour, dayofweek, when, from_json, to_json, struct
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType
)
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

KAFKA_BOOTSTRAP = "localhost:9092"
# Spark 4.x dùng Scala 2.13 -> phải dùng đúng artifact _2.13, nếu không sẽ ClassNotFound
SPARK_KAFKA_PKG = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"

spark = (
    SparkSession.builder
    .appName("FraudDetection_Streaming")
    .config("spark.jars.packages", SPARK_KAFKA_PKG)
    .config("spark.sql.shuffle.partitions", "4")
    # Trên macOS local mode: ép driver bind + advertise 127.0.0.1, tránh lỗi
    # "Failed to connect" khi máy quảng bá IP LAN
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark Session sẵn sàng! Version:", spark.version)

:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/leonnn/.ivy2.5.2/cache
The jars for the packages stored in: /Users/leonnn/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-63e93fb7-e1ee-4a8f-a6b6-411463242402;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org

Spark Session sẵn sàng! Version: 4.1.1


## 2. Định nghĩa schema cho giao dịch JSON
Kafka chuyển dữ liệu dạng byte, nên ta phải khai báo schema để parse JSON. Kiểu dữ liệu khớp với dữ liệu gốc mà model đã được train (xem `printSchema` trong notebook tiền xử lý).

In [3]:
schema = StructType([
    StructField("transaction_id", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("account_age_days", IntegerType()),
    StructField("total_transactions_user", IntegerType()),
    StructField("avg_amount_user", DoubleType()),
    StructField("amount", DoubleType()),
    StructField("country", StringType()),
    StructField("bin_country", StringType()),
    StructField("channel", StringType()),
    StructField("merchant_category", StringType()),
    StructField("promo_used", IntegerType()),
    StructField("avs_match", IntegerType()),
    StructField("cvv_result", IntegerType()),
    StructField("three_ds_flag", IntegerType()),
    StructField("transaction_time", StringType()),
    StructField("shipping_distance_km", DoubleType()),
    StructField("is_fraud", IntegerType()),
])
print("Đã định nghĩa schema với", len(schema.fields), "trường.")

Đã định nghĩa schema với 17 trường.


## 3. Đọc luồng từ Kafka (readStream)

In [4]:
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", "transactions")
    .option("startingOffsets", "latest")
    .load()
)

# Cột `value` là byte JSON -> cast string -> parse theo schema -> bung thành các cột
tx = (
    raw.select(from_json(col("value").cast("string"), schema).alias("d"))
       .select("d.*")
)
print("Đây có phải streaming DataFrame không?", tx.isStreaming)

Đây có phải streaming DataFrame không? True


## 4. Feature Engineering (lặp lại y hệt notebook batch)
Model được train với 5 đặc trưng dẫn xuất tạo NGOÀI pipeline. Stream phải tạo lại đúng các cột này trước khi đưa vào model, nếu không thứ tự/feature sẽ lệch.

In [5]:
tx = tx.withColumn(
    "transaction_hour", hour(col("transaction_time").cast("timestamp"))
)
tx = tx.withColumn(
    "transaction_dayofweek", dayofweek(col("transaction_time").cast("timestamp"))
)
tx = tx.withColumn(
    "country_match", when(col("country") == col("bin_country"), 1).otherwise(0)
)
tx = tx.withColumn(
    "is_weekend", when(col("transaction_dayofweek").isin([1, 7]), 1).otherwise(0)
)
tx = tx.withColumn(
    "amount_ratio", col("amount") / (col("avg_amount_user") + 1)
)
print("Feature engineering trên stream hoàn tất.")

Feature engineering trên stream hoàn tất.


## 5. Nạp model đã train và dự đoán
`PipelineModel` đã lưu chứa sẵn 4 StringIndexer + VectorAssembler + StandardScaler + RandomForest, nên chỉ cần `transform` trực tiếp lên stream.

In [6]:
# Model được notebook batch lưu lên HDFS (cell cuối của fraud_detection_sparkml.ipynb)
MODEL_PATH = "hdfs://localhost:9090/models/fraud_detection_rf"
print("Đang nạp model từ:", MODEL_PATH)
model = PipelineModel.load(MODEL_PATH)

predictions = model.transform(tx)
# Tách xác suất gian lận (phần tử thứ 2 của vector probability) ra cột số cho dễ đọc
predictions = predictions.withColumn(
    "fraud_prob", vector_to_array(col("probability"))[1]
)
print("Đã gắn model vào pipeline streaming.")

Đang nạp model từ: hdfs://localhost:9090/models/fraud_detection_rf


Đã gắn model vào pipeline streaming.


## 6. Lọc ra các giao dịch bị gắn cờ GIAN LẬN

In [7]:
alerts = predictions.filter(col("prediction") == 1.0).select(
    "transaction_id", "user_id", "amount", "merchant_category",
    "country", "bin_country",
    "is_fraud",          # nhãn thật (producer gửi kèm) -> đối chiếu độ chính xác
    "prediction",
    "fraud_prob"
)

## 7. Sink 1 — Hiển thị cảnh báo ngay trong notebook (foreachBatch)
Dùng `foreachBatch` để in cảnh báo của mỗi micro-batch ra ngay dưới cell này — trực quan khi thuyết trình.

In [8]:
def show_alerts(batch_df, batch_id):
    n = batch_df.count()
    if n > 0:
        correct = batch_df.filter(col("is_fraud") == 1).count()
        print(f"\n========== Micro-batch {batch_id}: {n} CẢNH BÁO GIAN LẬN ==========")
        print(f"(Trong đó {correct}/{n} thực sự là fraud theo nhãn gốc)")
        batch_df.orderBy(col("fraud_prob").desc()).show(n, truncate=False)

console_query = (
    alerts.writeStream
    .foreachBatch(show_alerts)
    .outputMode("append")
    .option("checkpointLocation", "/tmp/fraud_chk/console")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Sink hiển thị đã chạy. Bây giờ hãy chạy `python producer.py` ở terminal.")

26/06/12 20:49:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Sink hiển thị đã chạy. Bây giờ hãy chạy `python producer.py` ở terminal.


## 8. Sink 2 — Đẩy cảnh báo ngược ra Kafka topic `fraud-alerts`
Mô phỏng việc hệ thống hạ nguồn (chặn giao dịch / gửi thông báo) tiêu thụ cảnh báo realtime.

In [9]:
alert_kafka = alerts.select(
    col("transaction_id").cast("string").alias("key"),
    to_json(struct(
        "transaction_id", "user_id", "amount", "merchant_category",
        "country", "bin_country", "is_fraud", "prediction", "fraud_prob"
    )).alias("value")
)

kafka_query = (
    alert_kafka.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("topic", "fraud-alerts")
    .option("checkpointLocation", "/tmp/fraud_chk/kafka_alerts")
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start()
)
print("Sink Kafka `fraud-alerts` đã chạy.")

Sink Kafka `fraud-alerts` đã chạy.


26/06/12 20:49:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 9. (Tuỳ chọn) Sink 3 — Lưu cảnh báo ra HDFS dạng Parquet
Bỏ comment nếu muốn lưu trữ cảnh báo phục vụ phân tích/audit sau này.

In [13]:
parquet_query = (
    alerts.writeStream
    .format("parquet")
    .option("path", "hdfs://localhost:9090/fraud-alerts-parquet")
    .option("checkpointLocation", "/tmp/fraud_chk/parquet")
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .start()
)
print("Sink Parquet đã chạy.")

Sink Parquet đã chạy.


26/06/12 21:05:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/06/12 21:06:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/12 21:58:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 189008 ms exceeds timeout 120000 ms
26/06/12 21:58:21 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/12 21:58:27 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManage

## 10. Theo dõi & dừng các truy vấn streaming

In [11]:
# Trạng thái các query đang chạy
for q in spark.streams.active:
    print(f"{q.name or q.id} -> isActive={q.isActive}, status={q.status['message']}")

ec0778be-c1d3-407e-b6c3-430180412b16 -> isActive=True, status=Getting offsets from KafkaV2[Subscribe[transactions]]
53f3384a-cb72-4f71-ab43-87b6356f52e7 -> isActive=True, status=Getting offsets from KafkaV2[Subscribe[transactions]]
0a605811-61dc-4782-8942-36ddcf154f1b -> isActive=True, status=Initializing StreamExecution


In [12]:
# Dừng tất cả streaming query khi kết thúc demo
for q in spark.streams.active:
    q.stop()
print("Đã dừng toàn bộ streaming query.")
# spark.stop()  # bỏ comment nếu muốn tắt hẳn Spark Session

Đã dừng toàn bộ streaming query.


26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group 5eea27e5-97e1-459f-a166-973ba8d5bbaf. Cannot find active jobs for it.
26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group 5eea27e5-97e1-459f-a166-973ba8d5bbaf. Cannot find active jobs for it.
26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group c26e49a1-8435-4085-935b-50664a14a0a0. Cannot find active jobs for it.
26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group c26e49a1-8435-4085-935b-50664a14a0a0. Cannot find active jobs for it.
26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group bb5c2947-1f4c-4258-be57-2f36cf1dc034. Cannot find active jobs for it.
26/06/12 20:49:45 WARN DAGScheduler: Failed to cancel job group bb5c2947-1f4c-4258-be57-2f36cf1dc034. Cannot find active jobs for it.
